# 00 - Know your data before you model it

**Goal:** build intuition about your own series: how noisy, how missing, how much yesterday explains today.
Nothing here is scored. Everything here decides what is *worth* modelling.

**Concepts to learn first**
- *Time series*: order matters. A row is not independent of the previous row, so random shuffling lies to you.
- *Autocorrelation (ACF)*: the correlation of a series with itself `k` days earlier. High lag-1 means
  "tomorrow = today" is a strong baseline that a model must beat.
- *Missingness*: a day with no value is information (watch not worn). Is it random, or does it follow bad nights?
- *Outliers vs artifacts*: a 1.6 h "night" is usually a partial recording, not a real night (see the guard in `flags.py`).

Rule from `CLAUDE.md`: never silently fill missing days.

In [ ]:
# --- Setup (provided, working). Do not edit; fix things in api/app/ml/features.py instead. ---
import pathlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "api").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "api"))

from app.ml import features, evaluate          # the SAME modules the nightly job imports

raw = features.frame_from_csv(ROOT / "ml/data/daily_metrics.csv")   # one row per calendar day
feat = features.build(raw)                                          # baselines, z-scores, missing flags
print(f"{len(raw)} days, {raw.index.min().date()} -> {raw.index.max().date()}, "
      f"{int(raw['row_present'].sum())} with a row, {int(raw['sleep_hours'].isna().sum())} with no sleep value")
raw.tail()

## 1. Plot every metric over time
What do you see: trend, weekly rhythm, jumps, flat stretches? Mark the days you know were unusual.

In [ ]:
# TODO: one subplot per metric in features.METRICS (raw[m.key]). Gaps should show as gaps, not lines through them.

## 2. Missingness
How many days are missing per metric? Are the gaps clustered? Are they the same days across metrics?

In [ ]:
# TODO: raw[[m.key for m in features.METRICS]].isna() -> counts per metric, and a picture of *which* dates.

## 3. Distributions and outliers
Look at `sleep_hours`. Where is the mass? What is below 3 h? Decide: artifact or real? (Hint: compare `bb_max` that day.)

In [ ]:
# TODO: histogram of sleep_hours, and a table of the 5 lowest nights with their other metrics.

## 4. Autocorrelation
For `sleep_hours` and `hrv`: correlation with themselves at lag 1..7.

**Question:** if lag-1 is ~0.2, what does that say about how hard it is to beat `tomorrow = today`?

In [ ]:
# TODO: for lag in 1..7: raw['sleep_hours'].autocorr(lag)  (pandas ignores NaN pairs). Try hrv too. Plot as bars.

## 5. Weekly rhythm
Does day of week matter? (Use `raw.index.dayofweek`.) Beware: with ~16 weeks, each weekday has ~16 points.

In [ ]:
# TODO: mean and spread of sleep_hours by weekday; is the difference larger than the noise?

## 6. Does last night predict tomorrow's HRV?
Scatter `sleep_hours` (day t) vs `hrv` (day t+1). Correlation is not cause; note what else could drive both.

In [ ]:
# TODO: raw['sleep_hours'].shift(1) vs raw['hrv'] -> scatter and corr().

## My findings (write in your own words)
- What surprised me:
- Metric with the strongest lag-1 autocorrelation:
- Is missingness random?
- What I think is hardest to predict and why: